# Feature Engineering

# Reading Dataset

In [47]:
import pandas as pd

master_df = pd.read_csv("master_pds_dataset.csv")

print(master_df.shape)
master_df.head()

(499494, 46)


,distCode,distName,shopNo,month,year,rcNfsaAay,unitsNfsaAay,rcNfsaPhh,unitsNfsaPhh,totalRcNfsa,...,salt,otherShopTransCnt,officeCode,officeName,address,longitude,latitude,fpsStatus,fpsType,dateTime
0,532,Adilabad,1901001,7,2024,38,125,920,2957,958,...,0.0,36,532001.0,Talamadugu,DR Depo TALAMADUGU Talamadugu,78.391212,19.641643,Active,Normal Shop,2025-07-03 14:43:54.124224+05:30
1,532,Adilabad,1901002,7,2024,20,70,258,1070,278,...,0.0,28,532001.0,Talamadugu,Sunkari Ramesh Sunkidi,78.422012,19.669093,Active,Normal Shop,2025-07-03 14:43:54.124224+05:30
2,532,Adilabad,1901003,7,2024,17,60,158,596,175,...,0.0,22,532001.0,Talamadugu,Sakipelli Gajanan Umdam,78.427186,19.664645,Active,Normal Shop,2025-07-03 14:43:54.124224+05:30
3,532,Adilabad,1901004,7,2024,14,45,190,777,204,...,0.0,12,532001.0,Talamadugu,Rebbathi Rakesh Lingi,78.381016,19.696032,Active,Normal Shop,2025-07-03 14:43:54.124224+05:30
4,532,Adilabad,1901005,7,2024,155,520,530,2062,685,...,0.0,30,532001.0,Talamadugu,T Manohar Kuchulapur,78.351118,19.697063,Active,Normal Shop,2025-07-03 14:43:54.124224+05:30


## 🔥 Create Core Behavioral Features

"Raw transaction volumes are scale-dependent and influenced by district population. To evaluate distribution efficiency and participation behavior, I normalized key variables into ratios such as utilization rate, commodity intensity, and portability ratio. This allowed meaningful cross-district and temporal comparisons."


### 1️⃣ Participation Ratio, Transaction Ratio ,Utilization Ratio

In [ ]:
# to measure beneficiary coverage
master_df["participation_ratio"] = (
    master_df["noOfRcs"] / master_df["totalRcs"] 
)
# to measure operational behavior per participating card
master_df["transaction_intensity"] = (
    master_df["noOfTrans"] / master_df["noOfRcs"] 
)
# combines both coverage and intensity.
master_df["utilization_ratio"] = (
    master_df["noOfTrans"] / master_df["totalRcs"])



master_df[["utilization_ratio","participation_ratio", "transaction_intensity"]].describe()

,utilization_ratio,participation_ratio,transaction_intensity
count,499494.000000,499494.000000,499494.000000
mean,0.837087,0.834859,1.002705
std,0.190139,0.189476,0.010596
min,0.000824,0.000824,1.000000
25%,0.739130,0.737303,1.000000
50%,0.838449,0.836364,1.000000
75%,0.932507,0.930348,1.002674
max,4.192000,4.184000,2.000000


In the Telangana PDS analytics project, I engineered three key utilization metrics to evaluate shop-level performance.

First, I defined totalRcs as the total number of eligible ration cards registered under a shop in a given month.
noOfRcs represents the number of unique ration cards that actually transacted, and noOfTrans represents the total number of transaction events recorded.

Based on these, I created:

Participation Ratio (noOfRcs / totalRcs) to measure beneficiary coverage.

Transaction Intensity (noOfTrans / noOfRcs) to measure operational behavior per participating card.

Utilization Ratio (noOfTrans / totalRcs), which combines both coverage and intensity.

- The results showed that participation averaged around 83.5%, while transaction intensity was consistently around 1, indicating stable system behavior. Variability in utilization was primarily driven by participation differences rather than excessive transactions.
- Participation differences may arise due to seasonal migration, portability, socioeconomic factors, administrative lags, and accessibility constraints. Since transaction intensity per participating card is consistently around one, the variation is likely demand-driven rather than supply-driven.

Biggest insight: 
- The PDS system is operationally stable (1 transaction per card), but participation is below 100%, suggesting demand-side variability rather than supply-side inefficiency.

### District Level participation

In [49]:
master_df.groupby("distName")["participation_ratio"].mean().sort_values()

distName
Mahabubabad                0.699124
Wanaparthy                 0.740357
Nagarkarnool               0.743044
Mahbubnagar                0.756681
Suryapet                   0.758471
Nalgonda                   0.761043
Janagaon                   0.764570
Mulugu                     0.782979
Bhadrdri Kothagudem        0.783327
Warangal                   0.788375
Narayanpet                 0.796014
Yadadri Bhuvanagiri        0.799046
Vikarabad                  0.803521
Jayashankar Bhupalpalli    0.804199
Hanumakonda                0.815079
Khammam                    0.815668
Medak                      0.824500
Jogulamba Gadwal           0.833020
Peddapalli                 0.834266
Manchiryala                0.838103
Siddipet                   0.861328
Kamareddy                  0.869305
Sangareddy                 0.872856
Rajanna Siricilla          0.880276
Jagityal                   0.886575
Karimnagar                 0.890043
Kumarambheem Asifabad      0.892619
Nirmal             

- When I analyzed district-level participation ratios, I found significant variation across Telangana. Urban districts like Medchal and Hyderabad showed participation rates above 93%, while some rural districts like Mahabubabad were around 70%. Since transaction intensity per participating card remained stable across districts, the difference appears to be demand-driven — likely influenced by migration, agricultural self-sufficiency, or accessibility factors rather than supply-side inefficiencies.

- The 26% spread between lowest and highest participation districts indicates structural differences in beneficiary behavior. This suggests the need for district-specific policy interventions rather than uniform strategies.

### 2️⃣ Rice Distribution

In [50]:
# total rice distributed (shop-level)

master_df["total_rice"] = (
    master_df["riceAfsc"] +
    master_df["riceFsc"] +
    master_df["riceAap"]
)

In [51]:
master_df["total_rice"].describe()

count    499494.000000
mean       9117.857133
std        5934.281332
min           1.000000
25%        5518.000000
50%        7810.000000
75%       10855.000000
max      149655.000000
Name: total_rice, dtype: float64

Rice distribution varies significantly across shops, with a median of around 7,800 units per month but some large shops distributing up to 1.5 lakh units. This indicates structural differences in shop size and beneficiary load.

In [52]:
# Average rice distributed per beneficiary (per unit) in a shop-month.

master_df["rice_per_unit"] = (
    master_df["total_rice"] / master_df["totalUnits"]
)

master_df["rice_per_unit"].describe()

count    499494.000000
mean          5.477926
std           1.262091
min           0.000842
25%           4.800736
50%           5.482647
75%           6.137399
max          31.844595
Name: rice_per_unit, dtype: float64

In [53]:
# Total rows
total_rows = len(master_df)

# Rows where rice_per_unit > 10
high_rice_rows = master_df[master_df["rice_per_unit"] > 10]

# Count
high_count = len(high_rice_rows)

# Percentage
overall_percentage = (high_count / total_rows) * 100

print(f"Total rows: {total_rows}")
print(f"Rows with rice_per_unit > 10: {high_count}")
print(f"Overall Percentage: {overall_percentage:.2f}%")

Total rows: 499494
Rows with rice_per_unit > 10: 2988
Overall Percentage: 0.60%


0.6% of observations showed rice allocation above 10 kg per person.

#### District level >10 kg rice distribution

In [54]:
master_df[master_df["rice_per_unit"] > 10]["distName"].value_counts()

distName
Ranga Reddy                736
Sangareddy                 595
Medchal                    534
Hyderabad                  279
Karimnagar                 156
Rajanna Siricilla          154
Nalgonda                    92
Nirmal                      51
Nizamabad                   51
Peddapalli                  43
Kamareddy                   39
Suryapet                    39
Nagarkarnool                35
Khammam                     28
Manchiryala                 23
Yadadri Bhuvanagiri         23
Bhadrdri Kothagudem         19
Siddipet                    15
Mahabubabad                 14
Medak                       12
Adilabad                    10
Wanaparthy                   8
Jayashankar Bhupalpalli      7
Janagaon                     6
Vikarabad                    6
Jogulamba Gadwal             5
Mahbubnagar                  5
Jagityal                     2
Kumarambheem Asifabad        1
Name: count, dtype: int64

I found that only 0.6% of observations showed rice allocation above 10 kg per person. When I analyzed their distribution, they were concentrated in urban districts like Ranga Reddy, Medchal, and Hyderabad. This suggests that the elevated values are likely due to portability inflow under the One Nation One Ration Card system rather than systemic over-distribution.

### 3️⃣ Commodity Intensity (Rice to Wheat)

In [55]:
master_df["total_grain"] = master_df["total_rice"] + master_df["wheat"]

master_df["rice_share"] = (
    master_df["total_rice"] / master_df["total_grain"]
)
master_df["rice_share"].describe()

count    499494.000000
mean          0.991211
std           0.032982
min           0.054054
25%           1.000000
50%           1.000000
75%           1.000000
max           1.000000
Name: rice_share, dtype: float64

Telangana PDS is structurally rice-dominant, with rice accounting for approximately 99% of total grain distribution. Wheat distribution occurs in only a small fraction of shop-month observations and contributes minimally to overall grain composition.

Rice share was calculated as rice divided by total grain. The mean value was 0.99, meaning rice accounts for approximately 99% of grain distribution. The median being 1 indicates that in most shop-months, wheat distribution was zero. The very low standard deviation of 0.03 shows that this rice dominance is consistent across districts.


### 4️⃣ Portability Ratio

In [56]:
import numpy as np

master_df["portability_ratio"] = (
    master_df["otherShopTransCnt"] /
    master_df["noOfTrans"].replace(0, np.nan)
)
master_df["portability_ratio"].describe()

count    499494.000000
mean          0.283081
std           0.275898
min           0.000000
25%           0.054983
50%           0.158107
75%           0.484496
max           1.000000
Name: portability_ratio, dtype: float64

The average portability ratio was approximately 28%, meaning that nearly one-third of transactions at Fair Price Shops involved beneficiaries registered at other shops. However, the distribution was highly skewed, with some shops experiencing almost 100% portability inflow. This indicates strong migration-driven transaction patterns, especially in urban districts.

In [57]:
master_df.groupby("distName")[
    ["portability_ratio", "participation_ratio"]
].mean().sort_values("portability_ratio", ascending=False)

,portability_ratio,participation_ratio
distName,,
Hyderabad,0.775056,0.934281
Medchal,0.683601,0.956008
Ranga Reddy,0.360305,0.916460
Hanumakonda,0.354505,0.815079
Karimnagar,0.336884,0.890043
Nizamabad,0.327655,0.902972
Warangal,0.307706,0.788375
Manchiryala,0.286495,0.838103
Peddapalli,0.276336,0.834266


Portability is heavily concentrated in urban districts such as Hyderabad and Medchal, where it exceeds 65–75%. This spatial concentration is consistent with migration-driven transaction inflow under ONORC, rather than being uniformly distributed across the state.

### Average Monthly Transactions per Shop

In [58]:
# Average monthly workload of a shop
master_df["avg_monthly_transactions"] = (
    master_df
    .groupby(["distCode", "shopNo"])["noOfTrans"]
    .transform("mean")
)

In [59]:
master_df["avg_monthly_transactions"].describe()

count    499494.000000
mean        445.198261
std         265.929556
min           8.000000
25%         277.758621
50%         390.448276
75%         538.793103
max        5746.586207
Name: avg_monthly_transactions, dtype: float64

The average monthly transaction volume per shop is approximately 445 transactions, with a median of 390. The distribution is right-skewed, with a small number of high-volume urban shops handling several thousand transactions per month. This reflects significant capacity variation between rural and urban Fair Price Shops.

## 🔥 Step 3: Volatility Feature

In [60]:
#How much a shop’s annual transaction volume fluctuates across years.
#  Step 1 — Aggregate Yearly Transactions per Shop

yearly_trans = (
    master_df
    .groupby(["distCode", "shopNo", "year"])["noOfTrans"]
    .sum()
    .reset_index()
)

# Step 2 — Compute Std Across Years
volatility_df = (
    yearly_trans
    .groupby(["distCode", "shopNo"])["noOfTrans"]
    .std()
    .reset_index()
    .rename(columns={"noOfTrans": "yearly_transaction_volatility"})
)

# Step 3 — Merge Back

master_df = master_df.merge(
    volatility_df,
    on=["distCode", "shopNo"],
    how="left"
)



##### Coefficient of Variation (CV)


In [61]:
master_df = master_df.drop(
    columns=[
        "yearly_mean_transactions",
        "yearly_transaction_volatility",
        "cv"
    ],
    errors="ignore"
)

master_df = master_df.merge(
    yearly_stats,
    on=["distCode", "shopNo"],
    how="left"
)

In [62]:
# Relative variability as a percentage of average size.

# Step 1 — Aggregate yearly transaction totals per shop
yearly_trans = (
    master_df
    .groupby(["distCode", "shopNo", "year"], as_index=False)["noOfTrans"]
    .sum()
)

# Step 2 — Compute yearly mean and standard deviation per shop
yearly_stats = (
    yearly_trans
    .groupby(["distCode", "shopNo"], as_index=False)
    .agg(
        yearly_mean_transactions=("noOfTrans", "mean"),
        yearly_transaction_volatility=("noOfTrans", "std")
    )
)

# Step 3 — Compute coefficient of variation (CV)
yearly_stats["cv"] = (
    yearly_stats["yearly_transaction_volatility"] /
    yearly_stats["yearly_mean_transactions"]
)
master_df = master_df.merge(
    yearly_stats,
    on=["distCode", "shopNo"],
    how="left"
)

In [63]:
yearly_stats[[
    "yearly_transaction_volatility",
    "cv"
]].describe().round(2)

,yearly_transaction_volatility,cv
count,17319.00,17319.00
mean,1768.68,0.41
std,1063.25,0.05
min,0.00,0.00
25%,1104.64,0.40
50%,1563.80,0.41
75%,2156.08,0.42
max,19593.37,1.37


To assess long-term stability, I aggregated annual transaction totals per shop and computed their standard deviation across three years. The average absolute yearly volatility was approximately 1,769 transactions per shop, with a median of around 1,564, indicating moderate year-to-year fluctuation in transaction volume. However, since standard deviation is influenced by shop size, I also calculated the coefficient of variation (CV), which normalizes volatility relative to the shop’s average volume.

The mean and median CV were both approximately 41.5%, with a relatively small dispersion (standard deviation ≈ 5%). This suggests that relative variability is fairly consistent across shops, regardless of their size. In other words, both urban and rural shops experience similar proportional changes in yearly demand.

While most shops fall within a moderate volatility range, a few extreme cases show very high CV values (up to 1.37), indicating substantial structural shifts, potentially due to migration, shop consolidation, or policy-driven redistribution.

In [64]:
master_df.to_csv("master_pds_dataset.csv", index=False)
print("Master dataset saved successfully.")

Master dataset saved successfully.
